<a href="https://colab.research.google.com/github/aldo02032004/llm-topic-classification-pipeline/blob/main/Topic_Classification_Sentimen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. Install Dependency

In [1]:
import os, sys, shutil
from google.colab import userdata

GH_TOKEN = userdata.get('GH_TOKEN')
REPO_DIR = "/content/GREAT-Tools"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone -q https://{GH_TOKEN}@github.com/azmkto/GREAT-Tools.git {REPO_DIR}

for mod_name in list(sys.modules):
    if mod_name == "great" or mod_name.startswith("great."):
        del sys.modules[mod_name]

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

!pip uninstall -y torch torchvision torchaudio
!pip install -q torch torchvision torchaudio
!pip install -q -U google-genai pandas tqdm emoji requests openpyxl ftfy gliner transformers

print("[PENDING] Restart runtime sekarang: Runtime -> Restart session. Baru jalankan cell berikutnya.")

Found existing installation: torch 2.14.0
Uninstalling torch-2.14.0:
  Successfully uninstalled torch-2.14.0
Found existing installation: torchvision 0.29.0
Uninstalling torchvision-0.29.0:
  Successfully uninstalled torchvision-0.29.0
Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0
[PENDING] Restart runtime sekarang: Runtime -> Restart session. Baru jalankan cell berikutnya.


## 1. Import Library

In [2]:
import re
import json
import time
import random
import hashlib
import requests
import pandas as pd
from io import BytesIO
from datetime import date
from tqdm.auto import tqdm
from IPython.display import display
from google import genai
from google.genai import types
from google.colab import drive
from collections import Counter

from great.text import clean_for_bert
from great.labels import is_media_account

try:
    import emoji
    HAS_EMOJI_LIB = True
except ImportError:
    HAS_EMOJI_LIB = False
    print("[WARN] library 'emoji' tidak ada -> emoji akan dibuang, bukan dikonversi jadi teks.")

tqdm.pandas()

print("great loaded from:", __import__("great").__file__)
print("Commit terakhir  :", os.popen(f"git -C {REPO_DIR} rev-parse --short HEAD").read().strip())
print("Library siap.")

great loaded from: /content/GREAT-Tools/great/__init__.py
Commit terakhir  : bdd9a95
Library siap.


## 2. Set API Key Anthropic

Key diketik lewat input tersembunyi (tidak ke-log di notebook), jadi aman
kalau notebook ini nanti di-upload ke GitHub.

Alternatif lebih nyaman: pakai fitur **Secrets** Colab (ikon kunci di sidebar
kiri), simpan sebagai `ANTHROPIC_API_KEY`, lalu ganti isi cell ini dengan:

```python
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
```


In [3]:
os.environ["GOOOGLE_API_KEY"] = userdata.get("GOOOGLE_API_KEY")
gemini_client = genai.Client(api_key=os.environ["GOOOGLE_API_KEY"])
print("API key sudah di-set.")

API key sudah di-set.


# 3. TOPIK

In [4]:
PROJECT = "prabowo_vladivostok"
SOURCES = [
    "https://docs.google.com/spreadsheets/d/1rVvruuhS569La9-lm0qNiIAUoQp-DnnF/edit?usp=drive_link&ouid=116825097454650626545&rtpof=true&sd=true",
]
# Tema: key = nama tema, value = deskripsi untuk Gemini. "lainnya" wajib ada.
THEME_DESCRIPTIONS = {
    "kunjungan_prabowo": (
        "kunjungan Prabowo ke Rusia, kunjungan kenegaraan Prabowo, lawatan Prabowo, agenda Prabowo di Rusia, "
        "delegasi Indonesia ke Rusia, Prabowo temui Putin, pertemuan bilateral Prabowo Putin, "
        "Prabowo di Vladivostok, Prabowo di Moskow, Prabowo hadiri Eastern Economic Forum, "
        "Prabowo EEF, Prabowo forum ekonomi Rusia, kunjungan presiden RI ke Rusia, "
        "kerja sama Indonesia Rusia, nota kesepahaman Indonesia Rusia, MoU Indonesia Rusia, "
        "investasi Rusia ke Indonesia, kunjungan balasan Prabowo, protokoler kunjungan Prabowo, "
        "rombongan menteri dampingi Prabowo, jadwal kunjungan Prabowo Rusia, "
        "hasil pertemuan Prabowo Putin, pernyataan bersama Indonesia Rusia, "
        "kunjungan kenegaraan ke Kremlin, Prabowo di Kremlin, sambutan kenegaraan Prabowo Rusia, "
        "kereta kepresidenan Rusia, upacara penyambutan Prabowo, kunjungan luar negeri Prabowo, "
        "diplomasi Prabowo Rusia, agenda strategis Indonesia Rusia, "
        "lawatan kenegaraan Presiden Prabowo, kunjungan resmi Presiden Prabowo ke Rusia, "
        "Prabowo Subianto ke Rusia, Prabowo terbang ke Rusia, keberangkatan Prabowo ke Rusia, "
        "Prabowo tiba di Rusia, Prabowo pulang dari Rusia, kepulangan Prabowo dari Rusia, "
        "menteri luar negeri dampingi Prabowo, Sugiono dampingi Prabowo, Menlu RI ke Rusia, "
        "menteri pertahanan dampingi Prabowo, delegasi bisnis Indonesia Rusia, "
        "pengusaha Indonesia ikut kunjungan Rusia, pebisnis dampingi Prabowo, "
        "kesepakatan dagang Indonesia Rusia, perjanjian kerja sama Indonesia Rusia, "
        "kontrak dagang Indonesia Rusia, ekspor impor Indonesia Rusia, "
        "kerja sama pertahanan Indonesia Rusia, alutsista Rusia untuk Indonesia, "
        "kerja sama energi Indonesia Rusia, kerja sama nuklir Indonesia Rusia, "
        "PLTN Rusia Indonesia, Rosatom Indonesia, kerja sama pangan Indonesia Rusia, "
        "kunjungan Prabowo pasca kunjungan ke China, kunjungan Prabowo pasca KTT, "
        "reaksi publik kunjungan Prabowo Rusia, kritik kunjungan Prabowo ke Rusia, "
        "pujian kunjungan Prabowo ke Rusia, kontroversi kunjungan Prabowo Rusia, "
        "netralitas Indonesia kunjungan Rusia, politik luar negeri bebas aktif Prabowo, "
        "sikap Barat soal kunjungan Prabowo Rusia, respons AS soal kunjungan Prabowo Rusia, "
        "istana kepresidenan soal kunjungan Rusia, juru bicara presiden soal kunjungan Rusia, "
        "foto kunjungan Prabowo Rusia, video kunjungan Prabowo Rusia, momen Prabowo di Rusia, "
        "red carpet Prabowo Rusia, karpet merah Prabowo Rusia, penyambutan militer Prabowo Rusia"
    ),
    "rusia_putin": (
        "Rusia, Vladimir Putin, Presiden Rusia, Kremlin, Vladivostok, Moskow, Rusia Timur Jauh, "
        "Eastern Economic Forum, EEF Vladivostok, forum ekonomi Rusia, kebijakan luar negeri Rusia, "
        "hubungan Rusia dengan negara lain, sanksi terhadap Rusia, sanksi Barat ke Rusia, "
        "ekonomi Rusia, perdagangan Rusia, energi Rusia, gas Rusia, minyak Rusia, "
        "militer Rusia, angkatan bersenjata Rusia, perang Rusia Ukraina, konflik Rusia Ukraina, "
        "geopolitik Rusia, pernyataan Putin, pidato Putin, kebijakan Putin, "
        "Kementerian Luar Negeri Rusia, duta besar Rusia, kedutaan Rusia, "
        "kerja sama BRICS, Rusia BRICS, aliansi Rusia, mitra strategis Rusia, "
        "wilayah Timur Jauh Rusia, pelabuhan Vladivostok, industri Rusia, "
        "hubungan diplomatik dengan Rusia, kunjungan pejabat asing ke Rusia, "
        "Kremlin Moskow, Lapangan Merah, Istana Kremlin, juru bicara Kremlin, Dmitry Peskov, "
        "Sergey Lavrov, Menlu Rusia, diplomasi Rusia, Rusia dan negara Asia, Rusia dan ASEAN, "
        "Rusia dan Asia Tenggara, kunjungan kepala negara ke Rusia, tamu negara Rusia, "
        "ekonomi Rusia pasca sanksi, dampak sanksi terhadap Rusia, Rusia dan China, "
        "Rusia dan India, Rusia di panggung internasional, isolasi Rusia, Rusia G20"
    ),
    "lainnya": "topik di luar kategori di atas",
}

# Baris tanpa satu pun kata ini langsung "lainnya" (hemat Gemini + GLiNER).
# Kosongkan list = semua baris ke Gemini.
KW_RELEVAN = [
    "prabowo", "wowo", "psubianto", "presiden ri", "presiden indonesia",
    "rusia", "russia", "putin", "kremlin", "moskow", "moscow", "vladivostok",
    "eastern economic forum", "eef", "rosatom", "lavrov", "peskov", "brics", "ukraina",
]

# Nama lain -> nama baku (untuk laporan)
ENTITY_ALIASES = {
    "wowo": "Prabowo", "pak wowo": "Prabowo", "pak prabowo": "Prabowo",
    "prabowo subianto": "Prabowo", "psubianto": "Prabowo",
    "vlad putin": "Vladimir Putin", "putin": "Vladimir Putin", "vladimir putin": "Vladimir Putin",
    "vladivostock": "Vladivostok", "fefu": "Far Eastern Federal University",
}

# Akun yang dianggap media khusus di topik ini (tambahan dari daftar di library)
EXTRA_MEDIA_ACCOUNTS = {"kremlin", "kremlinru"}

# 4. Konfigurasi

Edit bagian ini kalau nama kolom, daftar tema, atau bobot ranking berubah.
Semua cell di bawah memakai variabel dari sini.


In [5]:
drive.mount('/content/drive')

COL_NO = "No"
COL_TYPE = "Type"
COL_HEADLINE = "Headline"
COL_MENTIONS = "Mentions"
COL_DATE = "Date"
COL_LINK = "Link"
COL_MEDIA = "Media"
COL_SENTIMENT = "Sentiment"
COL_AUTHOR_ID = "Author"
COL_FOLLOWERS = "Followers"
COL_RETWEETED = "Retweeted"
COL_FAVOURITED = "Favourited"

EXPECTED_COLS = [COL_NO, COL_TYPE, COL_HEADLINE, COL_MENTIONS, COL_DATE, COL_LINK,
                 COL_MEDIA, COL_SENTIMENT, COL_AUTHOR_ID, COL_FOLLOWERS,
                 COL_RETWEETED, COL_FAVOURITED]

THEMES = list(THEME_DESCRIPTIONS)
RELEVANT_THEMES = [t for t in THEMES if t != "lainnya"]

# --- Model & parameter LLM (Gemini buat tema + validasi NER + narasi) ---
GEMINI_MODEL = "gemini-3.1-flash-lite"
SLEEP_BETWEEN_CALLS = 4
GEN_CONFIG_DETERMINISTIC = dict(temperature=0.0, top_p=1.0, top_k=1)
THEME_CONFIDENCE_THRESHOLD = 0.6

# --- GLiNER config ---
GLINER_MODEL_NAME = "muchad/gliner-id"
GLINER_LABELS = ["lokasi", "instansi", "tokoh"]
GLINER_THRESHOLD = 0.5

# --- IndoBERT config ---
# Label map TIDAK di-hardcode, diambil dari config model di Step 6A.
INDOBERT_MODEL_NAME = "w11wo/indonesian-roberta-base-sentiment-classifier"

# --- Bobot skor ranking top author (harus berjumlah 1.0) ---
WEIGHT_POST_COUNT = 0.6
WEIGHT_ENGAGEMENT = 0.25
WEIGHT_FOLLOWERS = 0.15

BATCH_SIZE_CLASSIFY = 40
TOP_N_AUTHORS_PER_THEME = 10
MAX_POSTS_PER_AUTHOR_SUMMARY = 15

# --- Threshold review & ranking ---
RANKING_MIN_CONFIDENCE = 0.6      # tema di bawah ini tidak ikut ranking
NER_REVIEW_THRESHOLD = 0.7        # GLiNER di bawah ini dikirim ke Gemini
SENTIMENT_REVIEW_THRESHOLD = 0.7  # IndoBERT di bawah ini dikirim ke Gemini
REVIEW_SAMPLE_RATE = 0.05         # sampel acak confidence tinggi yang tetap dicek Gemini
DEDUP_PER_AUTHOR = False          # True = retweet dari author berbeda tetap dihitung (ukur penyebar)

# --- Pending training data (dipakai bersama semua topik, tiap baris bawa field project) ---
PENDING_PATHS = {
    "ner": "/content/drive/MyDrive/pending_ner_data.jsonl",
    "sentimen": "/content/drive/MyDrive/pending_sentiment_data.jsonl",
    "tema": "/content/drive/MyDrive/pending_tema_data.jsonl",
}
SAVE_PATH_AFTER_THEME = f"/content/drive/MyDrive/df_clean_after_theme_{PROJECT}.pkl"


def canonicalize_entity(name: str) -> str:
    key = name.strip().lower()
    return ENTITY_ALIASES.get(key, name.strip())

def canonicalize_list(names):
    return [canonicalize_entity(n) for n in names]


# --- Keyword prefilter ---
RE_RELEVAN = re.compile(
    r"(?<!\w)(?:" + "|".join(re.escape(k.lower()) for k in sorted(set(KW_RELEVAN), key=len, reverse=True)) + r")(?!\w)"
) if KW_RELEVAN else None


def keyword_match_theme(text):
    """'lainnya' kalau tidak ada keyword relevan sama sekali, selain itu None (ke Gemini)."""
    if RE_RELEVAN is None or RE_RELEVAN.search(str(text).lower()):
        return None
    return "lainnya"


def save_pending(task, rows):
    """Append ke pending. Hanya label dari Gemini/manual; output mentah model ditolak."""
    path = PENDING_PATHS[task]
    os.makedirs(os.path.dirname(path), exist_ok=True)
    n_ok = 0
    with open(path, "a", encoding="utf-8") as f:
        for r in rows:
            if not r.get("text") or r.get("label_source") not in ("gemini", "manual"):
                continue
            r.update(task=task, project=PROJECT, run_date=str(date.today()))
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
            n_ok += 1
    print(f"[pending] {task}: {n_ok}/{len(rows)} baris ditulis ke {path}")

print("Konfigurasi siap.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Konfigurasi siap.


# Step 1 — Load Data dari Google Sheets

Output: tabel mentah + daftar kolom, buat konfirmasi data ke-load dengan benar.


In [6]:
def resolve_xlsx_url(source: str) -> str:
    """Ubah link Google Sheets/Drive jadi url download .xlsx."""
    sheet_match = re.search(r"docs\.google\.com/spreadsheets/d/([a-zA-Z0-9-_]+)", source)
    drive_match = re.search(r"drive\.google\.com/file/d/([a-zA-Z0-9-_]+)", source)
    if sheet_match:
        return f"https://docs.google.com/spreadsheets/d/{sheet_match.group(1)}/export?format=xlsx"
    if drive_match:
        return f"https://drive.google.com/uc?export=download&id={drive_match.group(1)}"
    return source

def _download_bytes(url: str) -> BytesIO:
    session = requests.Session()
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = session.get(url, headers=headers, allow_redirects=True)
    resp.raise_for_status()
    content_type = resp.headers.get("Content-Type", "")
    if "html" in content_type.lower():
        raise ValueError(
            "Gagal download file asli (dapat HTML, bukan xlsx). "
            "Pastikan akses file/sheet sudah 'Anyone with the link'."
        )
    return BytesIO(resp.content)

def _find_best_sheet(xls: pd.ExcelFile, expected_cols: list, skiprows: int):
    best_sheet, best_match = xls.sheet_names[0], -1
    for name in xls.sheet_names:
        try:
            preview = pd.read_excel(xls, sheet_name=name, skiprows=skiprows, nrows=0)
            preview.columns = [c.strip() for c in preview.columns]
            match_count = sum(c in preview.columns for c in expected_cols)
            if match_count > best_match:
                best_match, best_sheet = match_count, name
        except Exception:
            continue
    return best_sheet

def load_raw_data(source: str, sheet_name=None, skiprows: int = 1) -> pd.DataFrame:
    url = resolve_xlsx_url(source)
    file_obj = _download_bytes(url) if url.startswith("http") else url
    xls = pd.ExcelFile(file_obj)

    if sheet_name is None:
        sheet_name = _find_best_sheet(xls, EXPECTED_COLS, skiprows)
        print(f"[INFO] Menggunakan sheet: '{sheet_name}'")

    df = pd.read_excel(xls, sheet_name=sheet_name, skiprows=skiprows)
    df.columns = [c.strip() for c in df.columns]

    missing = [c for c in EXPECTED_COLS if c not in df.columns]
    if missing:
        print(f"[WARNING] Kolom hilang: {missing}")

    available_cols = [c for c in EXPECTED_COLS if c in df.columns]
    return df[available_cols]

def load_raw_data_multi(sources: list, sheet_name=None, skiprows: int = 1) -> pd.DataFrame:
    """Load beberapa source lalu digabung jadi 1 df. Duplikat post yang muncul
    di lebih dari 1 file otomatis kebuang nanti di Step 3 (make_dedup_key),
    tidak perlu dedup manual di sini."""
    dfs = []
    for i, src in enumerate(sources):
        df_i = load_raw_data(src, sheet_name=sheet_name, skiprows=skiprows)
        df_i["source_file"] = f"file_{i+1}"
        print(f"[INFO] file_{i+1}: {df_i.shape}")
        dfs.append(df_i)
    df_combined = pd.concat(dfs, ignore_index=True)
    print(f"[INFO] Total gabungan: {df_combined.shape}")
    return df_combined

df_raw = load_raw_data_multi(SOURCES)
print(f"Data berhasil dimuat: {df_raw.shape}")
display(df_raw.head())

[INFO] Menggunakan sheet: 'Sheet1'
[INFO] file_1: (66891, 13)
[INFO] Total gabungan: (66891, 13)
Data berhasil dimuat: (66891, 13)


,No,Type,Headline,Mentions,Date,Link,Media,Sentiment,Author,Followers,Retweeted,Favourited,source_file
0,1,mention,"Netflix Rilis Skenario Sang Jenderal, Film Dok...",rmol news logo Film dokumenter Skenario Sang J...,2026-09-22 15:45:00,https://rmol.id/hiburan/read/2026/09/22/722557...,News,Negative,rmol.id,0,0,0,file_1
1,2,rt,NaN,"RT Negara hancur, Aturan baru OJK: Danantara, ...",2026-09-22 15:32:35,https://twitter.com/web/statuses/2102315108575...,Twitter,Negative,@iwantjiminp,5030,0,0,file_1
2,3,mention,Singapura Luncurkan Gerakan Baca 15 Menit Seha...,Presiden Prabowo Subianto berjabat tangan dan ...,2026-09-22 15:31:00,https://www.kompas.com/tren/read/2026/09/22/15...,News,Positive,www.kompas.com,0,0,0,file_1
3,4,mention,Singapura Luncurkan Gerakan Baca 15 Menit Seha...,Presiden Prabowo Subianto berjabat tangan dan ...,2026-09-22 15:31:00,https://www.kompas.com/tren/read/2026/09/22/15...,News,Positive,www.kompas.com,0,0,0,file_1
4,5,mention,Menkeu Ungkap Surplus BI Jadi Sumber Dana Luna...,Indonesia&#39;s Finance Minister Suahasil Naza...,2026-09-22 15:29:22,https://www.cnnindonesia.com/ekonomi/202609221...,News,Negative,www.cnnindonesia.com,0,0,0,file_1


# Step 2 - FILTER AKUN MEDIA (satu-satunya definisi + eksekusi)


In [7]:
df_raw[COL_FOLLOWERS] = pd.to_numeric(df_raw.get(COL_FOLLOWERS, 0), errors="coerce").fillna(0)
df_raw["is_media"] = [
    is_media_account(author, platform, extra_accounts=EXTRA_MEDIA_ACCOUNTS)
    for author, platform in zip(df_raw[COL_AUTHOR_ID], df_raw[COL_MEDIA])
]

print(f"[INFO] {df_raw['is_media'].sum()} baris media dibuang, {(~df_raw['is_media']).sum()} baris non-media lanjut")
display(df_raw[[COL_AUTHOR_ID, COL_MEDIA, "is_media"]].drop_duplicates(subset=COL_AUTHOR_ID).head(20))

df_raw = df_raw[~df_raw["is_media"]].drop(columns=["is_media"]).reset_index(drop=True)

[INFO] 28691 baris media dibuang, 38200 baris non-media lanjut


,Author,Media,is_media
0,rmol.id,News,True
1,@iwantjiminp,Twitter,False
2,www.kompas.com,News,True
4,www.cnnindonesia.com,News,True
5,@wanzhouvers,Twitter,False
6,economy.okezone.com,News,True
8,index.okezone.com,News,True
10,@moodmasyarakat,Twitter,False
11,wartaekonomi.co.id,News,True
16,nasional.tvrinews.com,News,True


# Step 3 - CLEANING TEKS

In [8]:
RE_RT_PREFIX = re.compile(r"^\s*RT\s*@[A-Za-z0-9_]+\s*:\s*", flags=re.IGNORECASE)
RE_ELONGATION = re.compile(r"(.)\1{2,}")
RE_MULTI_SPACE = re.compile(r"\s+")
RE_URL = re.compile(r"https?://\S+|www\.\S+")

def fix_elongation(text: str) -> str:
    return RE_ELONGATION.sub(r"\1\1", text)

def demojize_or_strip(text: str) -> str:
    if HAS_EMOJI_LIB:
        text = emoji.demojize(text, language="id" if "id" in emoji.LANGUAGES else "en")
        return text.replace("_", " ").replace(":", " ")
    return text

def build_raw_text(headline, mentions) -> str:
    headline = "" if pd.isna(headline) else str(headline).strip()
    mentions = "" if pd.isna(mentions) else str(mentions).strip()
    if not headline or headline.lower() == mentions.lower() or headline.lower() == "nan":
        return mentions or headline
    if not mentions:
        return headline
    return f"{headline}. {mentions}"

def clean_text(raw: str) -> str:
    if not isinstance(raw, str) or not raw.strip():
        return ""
    text = RE_RT_PREFIX.sub("", raw)
    text = demojize_or_strip(text)
    text = fix_elongation(text)
    text = clean_for_bert(text)
    return RE_MULTI_SPACE.sub(" ", text).strip()

def clean_for_ner(raw: str) -> str:
    """Versi teks untuk GLiNER: huruf besar-kecil DIPERTAHANKAN, slang tidak diubah.
    Kapitalisasi adalah petunjuk kuat nama orang/tempat."""
    if not isinstance(raw, str) or not raw.strip():
        return ""
    text = RE_RT_PREFIX.sub("", raw)
    text = RE_URL.sub(" ", text)
    text = demojize_or_strip(text)
    text = fix_elongation(text)
    text = text.replace("#", " ").replace("@", " ")
    return RE_MULTI_SPACE.sub(" ", text).strip()

def make_dedup_key(text_clean: str) -> str:
    key = re.sub(r"[^\w\s]", "", text_clean.lower())
    key = RE_MULTI_SPACE.sub(" ", key).strip()
    return hashlib.md5(key.encode("utf-8")).hexdigest()

headline_col = df_raw[COL_HEADLINE] if COL_HEADLINE in df_raw.columns else pd.Series([""] * len(df_raw))
mentions_col = df_raw[COL_MENTIONS] if COL_MENTIONS in df_raw.columns else pd.Series([""] * len(df_raw))
df_raw["text_raw_combined"] = [build_raw_text(h, m) for h, m in zip(headline_col, mentions_col)]

df_raw["text_clean"] = df_raw["text_raw_combined"].astype(str).progress_apply(clean_text)
df_raw["text_ner"] = df_raw["text_raw_combined"].astype(str).apply(clean_for_ner)

before = len(df_raw)
df_clean = df_raw[df_raw["text_clean"].str.split().str.len().fillna(0) >= 3].copy()
print(f"[INFO] Buang {before - len(df_clean)} baris kosong/terlalu pendek")

df_clean["dedup_key"] = df_clean["text_clean"].apply(make_dedup_key)
dedup_subset = ["dedup_key", COL_AUTHOR_ID] if DEDUP_PER_AUTHOR else ["dedup_key"]
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=dedup_subset, keep="first").drop(columns=["dedup_key"])
df_clean = df_clean.reset_index(drop=True)
print(f"[INFO] Buang {before - len(df_clean)} duplikat (per author: {DEDUP_PER_AUTHOR})")
print(f"[OK] {len(df_clean)} baris tersisa setelah cleaning")

display(df_clean[["text_raw_combined", "text_clean", "text_ner"]].head(5))

  0%|          | 0/38200 [00:00<?, ?it/s]

[INFO] Buang 1670 baris kosong/terlalu pendek
[INFO] Buang 13892 duplikat (per author: False)
[OK] 22638 baris tersisa setelah cleaning


,text_raw_combined,text_clean,text_ner
0,"RT Negara hancur, Aturan baru OJK: Danantara, ...","RT Negara hancur, Aturan baru OJK Danantara, B...","RT Negara hancur, Aturan baru OJK Danantara, B..."
1,"RT ""Prabowo NPD this NPD that.."" . . . .. tapi...","RT ""Prabowo NPD this NPD that.."" . . . .. tapi...","RT ""Prabowo NPD this NPD that.."" . . . .. tapi..."
2,Pakar Hukum UNPAD Romli Atmasasmita: Presiden ...,Pakar Hukum UNPAD Romli Atmasasmita Presiden P...,Pakar Hukum UNPAD Romli Atmasasmita Presiden P...
3,RT P2G Desak MBG Dihentikan: Guru Jangan Terus...,RT P2G Desak MBG Dihentikan Guru Jangan Terus ...,RT P2G Desak MBG Dihentikan Guru Jangan Terus ...
4,"emang prabowo sialan, mbg sialan [RE kenhans03]","memang prabowo sialan, mbg sialan [RE kenhans03]","emang prabowo sialan, mbg sialan [RE kenhans03]"


## Step 4A - Load GLiNER

In [9]:
from gliner import GLiNER

gliner_model = GLiNER.from_pretrained(GLINER_MODEL_NAME)
print("GLiNER siap.")

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

GLiNER siap.


## Step 4B - GLiNER extract entitas mentah

In [10]:
def gliner_extract_entities(text):
    """Return (entities dict surface form, min confidence).
    Surface form = persis seperti di teks, TIDAK di-canonicalize.
    Canonical hanya untuk laporan, bukan untuk training."""
    ents = gliner_model.predict_entities(text, GLINER_LABELS, threshold=GLINER_THRESHOLD)
    result = {lab: [] for lab in GLINER_LABELS}
    scores = []
    for e in ents:
        lab = e["label"]
        if lab in result and e["text"] not in result[lab]:
            result[lab].append(e["text"])
            scores.append(e["score"])
    return result, (min(scores) if scores else None)

## Step 4C - Gemini: tema + validasi NER (prompt beda dari versi lama, sekarang audit bukan ekstrak)

In [11]:
THEME_LIST_STR = "\n".join(f"- {t}: {THEME_DESCRIPTIONS[t]}" for t in THEMES)

def build_validation_prompt(batch_items):
    """batch_items: list of (text, gliner_entities_dict)"""
    numbered = []
    for i, (text, ents) in enumerate(batch_items):
        numbered.append(
            f'{i+1}. Teks: "{text}"\n'
            f'   Prediksi GLiNER -> lokasi: {ents["lokasi"]}, instansi: {ents["instansi"]}, tokoh: {ents["tokoh"]}'
        )
    joined = "\n".join(numbered)

    return f"""Kamu classifier tema + validator NER cuitan bahasa Indonesia.

Daftar tema:
{THEME_LIST_STR}

Untuk SETIAP item: tentukan tema (primary_theme + confidence), DAN validasi prediksi GLiNER.
Koreksi kalau salah kategori, tambah kalau ada entitas yang kelewat, hapus kalau itu bukan entitas asli
(contoh: emoji jadi teks, jabatan tanpa nama, judul buku).

ATURAN PENULISAN ENTITAS (WAJIB):
- Tulis entitas PERSIS seperti muncul di teks: ejaan, singkatan, huruf besar-kecil sama.
- JANGAN dinormalisasi atau dilengkapi. Teks "wowo tiba di vladivostok" -> tokoh ["wowo"],
  lokasi ["vladivostok"]. BUKAN ["Prabowo"] atau ["Vladivostok, Rusia"].
- Jabatan tanpa nama ("presiden", "menlu") bukan tokoh.

Item:
{joined}

Jawab HANYA dengan JSON array, tanpa penjelasan, tanpa markdown code block.
Setiap objek WAJIB punya "index" sesuai nomor item:
[
  {{"index": 1, "primary_theme": "{THEMES[0]}", "confidence": 0.9,
    "lokasi": [], "instansi": [], "tokoh": []}}
]
Jumlah item HARUS sama persis dengan jumlah teks ({len(batch_items)} item)."""


def entities_dict_to_spans(text: str, entities: dict):
    """{"lokasi": [surface,...], ...} -> ([[start, end, label], ...], [nama tidak ketemu]).
    Semua kemunculan, tanpa peduli huruf besar-kecil, kata utuh.
    Nama panjang diproses duluan supaya 'Vladivostok Rusia' menang atas 'Vladivostok'."""
    pairs = sorted({(n.strip(), lab) for lab, names in entities.items() for n in names if n and n.strip()},
                   key=lambda p: len(p[0]), reverse=True)
    spans, missing = [], []
    for name, label in pairs:
        pat = re.compile(r"(?<!\w)" + re.escape(name) + r"(?!\w)", re.IGNORECASE)
        found = False
        for m in pat.finditer(text):
            found = True
            if not any(m.start() < e and s < m.end() for s, e, _ in spans):
                spans.append([m.start(), m.end(), label])
        if not found:
            missing.append(name)
    return sorted(spans), missing


def parse_json_safe(raw_text):
    """Parse JSON, abaikan trailing garbage setelah array/object pertama yang valid."""
    decoder = json.JSONDecoder()
    obj, _ = decoder.raw_decode(raw_text.strip())
    return obj


api_call_count = 0

def is_quota_error(e) -> bool:
    return "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e)


def gemini_json(prompt, max_output_tokens):
    """Satu call Gemini deterministik, hasil sudah di-parse jadi JSON."""
    global api_call_count
    api_call_count += 1
    resp = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            max_output_tokens=max_output_tokens,
            **GEN_CONFIG_DETERMINISTIC,
        ),
    )
    return parse_json_safe(resp.text)


def order_by_index(parsed, n):
    """List objek ber-field 'index' (1..n) -> list urut. Error kalau ada index yang hilang."""
    by_idx = {int(it["index"]): it for it in parsed if isinstance(it, dict) and "index" in it}
    if set(by_idx) != set(range(1, n + 1)):
        raise ValueError(f"Index hasil tidak lengkap: dapat {sorted(by_idx)}, harus 1..{n}")
    return [by_idx[i] for i in range(1, n + 1)]


def validate_batch_with_gemini(batch_items, retry=6):
    """Return (results urut sesuai batch_items, success).
    Hasil dicocokkan pakai field 'index', BUKAN posisi."""
    prompt = build_validation_prompt(batch_items)
    for attempt in range(retry):
        try:
            return order_by_index(gemini_json(prompt, 8000), len(batch_items)), True
        except Exception as e:
            err = str(e)
            if is_quota_error(e):
                raise
            wait = 20 * (attempt + 1) if ("503" in err or "UNAVAILABLE" in err) else 5 * (attempt + 1)
            print(f"[WARN] gagal validasi ({attempt+1}/{retry}): {e} -> tunggu {wait}s")
            time.sleep(wait)

    print("[ERROR] Batch validasi gagal total. Entity tetap dari GLiNER, baris TIDAK masuk training.")
    return [{} for _ in batch_items], False

## Step 4D - Eksekusi: prefilter -> GLiNER -> Gemini validasi

In [12]:
for col in ("theme", "theme_confidence", "entities_lokasi", "entities_instansi", "entities_tokoh", "ner_source"):
    df_clean[col] = None
df_clean["ner_failed"] = False
df_clean["from_prefilter"] = False
df_clean["gliner_corrected"] = False

# tahap 1: keyword prefilter -> tema saja
sig = df_clean["text_clean"].apply(keyword_match_theme)
mask_sig = sig.notna()
df_clean.loc[mask_sig, "theme"] = sig[mask_sig]
df_clean.loc[mask_sig, "theme_confidence"] = 1.0
df_clean.loc[mask_sig, "from_prefilter"] = True
need_theme_idx = df_clean.index[df_clean["theme"].isna()].tolist()
print(f"[INFO] Prefilter: {mask_sig.sum()} baris dapat tema, {len(need_theme_idx)} ambigu")

# tahap 2: GLiNER di semua baris ambigu + baris relevan dari prefilter
ner_idx = df_clean.index[df_clean["theme"].isna() | df_clean["theme"].isin(RELEVANT_THEMES)].tolist()
gliner_results = {}
for idx in tqdm(ner_idx, desc="GLiNER NER"):
    ents, min_conf = gliner_extract_entities(df_clean.at[idx, "text_ner"])
    gliner_results[idx] = {"entities": ents, "min_conf": min_conf}
    for lab in GLINER_LABELS:
        df_clean.at[idx, f"entities_{lab}"] = canonicalize_list(ents[lab])
    df_clean.at[idx, "ner_source"] = "gliner"

for idx in df_clean.index.difference(ner_idx):
    for lab in GLINER_LABELS:
        df_clean.at[idx, f"entities_{lab}"] = []

# tahap 3: pilih baris untuk Gemini = tema ambigu + NER ragu + sampel acak
prefilter_relevant = [i for i in ner_idx if df_clean.at[i, "from_prefilter"]]
low_conf_idx = [i for i in prefilter_relevant
                if gliner_results[i]["min_conf"] is not None
                and gliner_results[i]["min_conf"] < NER_REVIEW_THRESHOLD]
low_conf_set = set(low_conf_idx)
rest = [i for i in prefilter_relevant if i not in low_conf_set]
sampled_idx = random.sample(rest, int(len(rest) * REVIEW_SAMPLE_RATE)) if rest else []
need_theme_set = set(need_theme_idx)
review_idx = need_theme_idx + low_conf_idx + sampled_idx
print(f"[INFO] Ke Gemini: {len(need_theme_idx)} tema ambigu, {len(low_conf_idx)} NER ragu, "
      f"{len(sampled_idx)} sampel acak")

# tahap 4: Gemini validasi
n_batches = -(-len(review_idx) // BATCH_SIZE_CLASSIFY)
print(f"[INFO] Estimasi {n_batches} request (~{n_batches * SLEEP_BETWEEN_CALLS / 60:.1f} menit)")
ner_training_data, tema_training_data = [], []
n_skip_missing = 0

for b in tqdm(range(n_batches), desc="Validate NER+Tema"):
    idx_batch = review_idx[b * BATCH_SIZE_CLASSIFY:(b + 1) * BATCH_SIZE_CLASSIFY]
    batch_items = [(df_clean.at[i, "text_ner"], gliner_results[i]["entities"]) for i in idx_batch]

    try:
        results, success = validate_batch_with_gemini(batch_items)
    except Exception as e:
        if is_quota_error(e):
            print(f"[STOP] Limit harian habis di batch {b}/{n_batches}. Total API call: {api_call_count}")
            break
        raise

    for r, df_idx in zip(results, idx_batch):   # aman: results sudah diurutkan by index
        if not success:
            df_clean.at[df_idx, "ner_failed"] = True
            continue   # tema tetap None (terhitung belum diproses), entity dari GLiNER, tidak masuk training

        if df_idx in need_theme_set:
            theme = r.get("primary_theme") if r.get("primary_theme") in THEMES else "lainnya"
            conf = float(r.get("confidence", 0.5))
            df_clean.at[df_idx, "theme"] = theme
            df_clean.at[df_idx, "theme_confidence"] = conf
            tema_training_data.append({
                "text": df_clean.at[df_idx, "text_clean"], "label": theme,
                "confidence": conf, "label_source": "gemini",
            })

        ents = {lab: [x for x in (r.get(lab) or []) if isinstance(x, str)] for lab in GLINER_LABELS}
        for lab in GLINER_LABELS:
            df_clean.at[df_idx, f"entities_{lab}"] = canonicalize_list(ents[lab])
        df_clean.at[df_idx, "ner_source"] = "gemini"

        gl = gliner_results[df_idx]["entities"]
        df_clean.at[df_idx, "gliner_corrected"] = any(
            sorted(x.lower() for x in gl[lab]) != sorted(x.lower() for x in ents[lab]) for lab in GLINER_LABELS
        )

        text_ner = df_clean.at[df_idx, "text_ner"]
        spans, missing = entities_dict_to_spans(text_ner, ents)
        if missing:
            n_skip_missing += 1   # label parsial = racun training, jangan dipakai
            continue

        reason = ("tema_ambigu" if df_idx in need_theme_set
                  else "low_conf" if df_idx in low_conf_set else "sample")
        ner_training_data.append({
            "text": text_ner, "entities": spans, "label_source": "gemini",
            "review_reason": reason, "model_version": GLINER_MODEL_NAME,
            "last_confidence": gliner_results[df_idx]["min_conf"],
        })

    time.sleep(SLEEP_BETWEEN_CALLS)

df_clean["needs_review"] = df_clean["theme_confidence"].astype(float) < THEME_CONFIDENCE_THRESHOLD

validated = df_clean["ner_source"] == "gemini"
if validated.sum() > 0:
    rate = df_clean.loc[validated, "gliner_corrected"].astype(bool).mean()
    print(f"\n[INFO] correction_rate GLiNER: {100 * rate:.1f}% dari {validated.sum()} baris tervalidasi")
print(f"[INFO] {n_skip_missing} baris tidak masuk training (entity Gemini tidak ditemukan di teks)")

n_unprocessed = df_clean["theme"].isna().sum()
if n_unprocessed:
    print(f"[WARN] {n_unprocessed} baris BELUM punya tema (quota habis / batch gagal). "
          f"Laporan di bawah TIDAK lengkap.")

print("\nDistribusi tema:")
display(df_clean["theme"].value_counts(dropna=False))

print("\nContoh hasil klasifikasi + NER:")
display(df_clean.loc[df_clean["theme"].isin(RELEVANT_THEMES),
        ["text_clean", "theme", "ner_source", "entities_lokasi", "entities_instansi", "entities_tokoh"]].head(10))

df_clean.to_pickle(SAVE_PATH_AFTER_THEME)
print(f"[OK] df_clean tersimpan ke {SAVE_PATH_AFTER_THEME}. Total API call: {api_call_count}")

save_pending("ner", ner_training_data)
save_pending("tema", tema_training_data)

[INFO] Prefilter: 9963 baris dapat tema, 12675 ambigu


GLiNER NER:   0%|          | 0/12675 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/gliner/data_processing/processor.py:427: UserWarning: Sentence of length 411 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/usr/local/lib/python3.13/dist-packages/gliner/data_processing/processor.py:427: UserWarning: Sentence of length 470 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


KeyboardInterrupt: 

# Step 5 - RANKING TOP AUTHOR PER TEMA

Hanya akun non-media (sudah difilter Step 2). Skor = post count + engagement + followers.

In [ ]:
processed = df_clean["theme"].notna()
relevant = processed & (df_clean["theme"] != "lainnya")
confident = df_clean["theme_confidence"].astype(float) >= RANKING_MIN_CONFIDENCE

df_topic = df_clean[relevant & confident].copy()
print(f"[INFO] {len(df_topic)} baris ikut ranking, "
      f"{(relevant & ~confident).sum()} baris dibuang karena tema ragu (< {RANKING_MIN_CONFIDENCE})")
if (~processed).sum():
    print(f"[WARN] {(~processed).sum()} baris belum diproses. Ranking PARSIAL.")

for col in (COL_FAVOURITED, COL_RETWEETED, COL_FOLLOWERS):
    df_topic[col] = pd.to_numeric(df_topic[col], errors="coerce").fillna(0) if col in df_topic else 0

df_topic["engagement"] = df_topic[COL_FAVOURITED] + df_topic[COL_RETWEETED]

top_authors_per_theme = {}

for theme in RELEVANT_THEMES:
    sub = df_topic[df_topic["theme"] == theme]
    if sub.empty:
        print(f"[INFO] Tidak ada data untuk tema '{theme}'")
        continue

    agg = (
        sub.groupby(COL_AUTHOR_ID)
        .agg(
            jumlah_post=(COL_AUTHOR_ID, "count"),
            total_engagement=("engagement", "sum"),
            followers=(COL_FOLLOWERS, "max"),
        )
        .reset_index()
        .rename(columns={COL_AUTHOR_ID: "author_id"})
    )

    max_post = agg["jumlah_post"].max() or 1
    max_eng = agg["total_engagement"].max() or 1
    max_followers = agg["followers"].max() or 1
    agg["score"] = (
        WEIGHT_POST_COUNT * (agg["jumlah_post"] / max_post)
        + WEIGHT_ENGAGEMENT * (agg["total_engagement"] / max_eng)
        + WEIGHT_FOLLOWERS * (agg["followers"] / max_followers)
    )

    agg = agg.sort_values("score", ascending=False).head(TOP_N_AUTHORS_PER_THEME)
    agg.insert(0, "theme", theme)
    agg["rank"] = range(1, len(agg) + 1)
    top_authors_per_theme[theme] = agg

    print(f"\n=== Top Author: {theme} ===")
    display(agg[["rank", "author_id", "jumlah_post", "total_engagement", "followers", "score"]])

df_top_authors = (
    pd.concat(top_authors_per_theme.values(), ignore_index=True)
    if top_authors_per_theme else
    pd.DataFrame(columns=["theme", "author_id", "jumlah_post", "total_engagement", "followers", "score", "rank"])
)

## Step 6A - Load IndoBERT sentimen

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

indobert_tokenizer = AutoTokenizer.from_pretrained(INDOBERT_MODEL_NAME)
indobert_model = AutoModelForSequenceClassification.from_pretrained(INDOBERT_MODEL_NAME)
indobert_model.eval()

# label map diambil dari config model, bukan hardcode (urutan label tiap model beda)
LABEL_NORMALIZE = {"positive": "positif", "negative": "negatif", "neutral": "netral",
                   "positif": "positif", "negatif": "negatif", "netral": "netral"}
INDOBERT_LABEL_MAP = {int(i): LABEL_NORMALIZE.get(str(l).lower(), str(l).lower())
                      for i, l in indobert_model.config.id2label.items()}
print("[INFO] Label map dari config model:", INDOBERT_LABEL_MAP)
if not {"positif", "negatif", "netral"} <= set(INDOBERT_LABEL_MAP.values()):
    raise ValueError("id2label tidak dikenali (mis. LABEL_0). Cek model card, isi INDOBERT_LABEL_MAP manual.")


def indobert_predict_sentiment(text):
    inputs = indobert_tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    with torch.no_grad():
        logits = indobert_model(**inputs).logits
    probs = torch.softmax(logits, dim=1)[0]
    pred_id = int(torch.argmax(probs))
    return INDOBERT_LABEL_MAP[pred_id], float(probs[pred_id]), probs.tolist()

print("IndoBERT siap.")

## Step 6B - IndoBERT prediksi per-post, agregasi ke level author

In [ ]:
def aggregate_author_sentiment(post_sentiments):
    """post_sentiments: list of (label, confidence) per post.
    kontroversi = campuran positif & negatif signifikan. netral = mayoritas netral / tanpa sikap."""
    counts = Counter(label for label, _ in post_sentiments)
    total = sum(counts.values())
    if total == 0:
        return "netral", 0.0
    pos_ratio = counts.get("positif", 0) / total
    neg_ratio = counts.get("negatif", 0) / total
    net_ratio = counts.get("netral", 0) / total

    if pos_ratio >= 0.3 and neg_ratio >= 0.3:
        return "kontroversi", max(pos_ratio, neg_ratio)
    if net_ratio >= 0.5 or pos_ratio == neg_ratio:
        return "netral", net_ratio
    if pos_ratio > neg_ratio:
        return "positif", pos_ratio
    return "negatif", neg_ratio


def label_sentiment_batch_gemini(items, retry=4):
    """items: list of (text, theme). Return list label per item (None kalau gagal).
    Label per POST terhadap TEMA. 'kontroversi' bukan label post, itu kesimpulan level author."""
    numbered = "\n".join(f'{i+1}. Tema: {th.replace("_", " ")}\n   Teks: "{t}"'
                         for i, (t, th) in enumerate(items))
    prompt = f"""Kamu pelabel sentimen cuitan bahasa Indonesia.
Untuk SETIAP teks, tentukan sikap penulis TERHADAP TEMA yang disebut. Pilih satu:
- positif: mendukung, memuji, optimis terhadap tema
- negatif: mengkritik, menolak, pesimis terhadap tema
- netral: informasi tanpa sikap, atau sikap tidak jelas

Item:
{numbered}

Jawab HANYA JSON array, tanpa penjelasan: [{{"index": 1, "label": "netral"}}]
Jumlah item HARUS {len(items)}."""
    n = len(items)
    for attempt in range(retry):
        try:
            labels = [x.get("label") for x in order_by_index(gemini_json(prompt, 2000), n)]
            return [lab if lab in ("positif", "negatif", "netral") else None for lab in labels]
        except Exception as e:
            if is_quota_error(e):
                raise
            wait = 5 * (attempt + 1)
            print(f"[WARN] gagal label sentimen ({attempt+1}/{retry}): {e} -> tunggu {wait}s")
            time.sleep(wait)
    return [None] * n


pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

author_sentiment_results = {}
review_queue = {}   # (text, theme) -> (confidence, reason), dedup otomatis

for _, row in df_top_authors.iterrows():
    author_id, theme = row["author_id"], row["theme"]

    author_posts_df = (
        df_topic[(df_topic[COL_AUTHOR_ID] == author_id) & (df_topic["theme"] == theme)]
        .sort_values("engagement", ascending=False)   # post paling berpengaruh duluan
    )
    posts = author_posts_df["text_clean"].dropna().tolist()[:MAX_POSTS_PER_AUTHOR_SUMMARY]
    if not posts:
        continue

    post_sentiments = []
    for text in posts:
        label, conf, _ = indobert_predict_sentiment(text)
        post_sentiments.append((label, conf))
        if conf < SENTIMENT_REVIEW_THRESHOLD:
            review_queue.setdefault((text, theme), (conf, "low_conf"))
        elif random.random() < REVIEW_SAMPLE_RATE:
            review_queue.setdefault((text, theme), (conf, "sample"))

    agg_label, agg_confidence = aggregate_author_sentiment(post_sentiments)
    author_sentiment_results[(author_id, theme)] = {
        "sentiment": agg_label,
        "confidence": agg_confidence,
        "post_sentiments": post_sentiments,
        "posts": posts,
    }

print(f"[INFO] IndoBERT selesai prediksi {len(author_sentiment_results)} author")

# label koreksi per post dari Gemini -> training data sentimen
sentiment_training_data = []
queue_items = list(review_queue.items())
print(f"[INFO] {len(queue_items)} post dikirim ke Gemini untuk label sentimen")

for b in range(0, len(queue_items), BATCH_SIZE_CLASSIFY):
    chunk = queue_items[b:b + BATCH_SIZE_CLASSIFY]
    try:
        labels = label_sentiment_batch_gemini([key for key, _ in chunk])
    except Exception as e:
        if is_quota_error(e):
            print(f"[STOP] Quota habis saat label sentimen. {len(sentiment_training_data)} label tersimpan.")
            break
        raise
    for ((text, theme), (conf, reason)), label in zip(chunk, labels):
        if label is None:
            continue
        sentiment_training_data.append({
            "text": text, "theme": theme, "label": label, "label_source": "gemini",
            "review_reason": reason, "model_version": INDOBERT_MODEL_NAME, "last_confidence": conf,
        })
    time.sleep(SLEEP_BETWEEN_CALLS)

save_pending("sentimen", sentiment_training_data)

## Step 6C - Gemini generate ringkasan naratif (bukan nentuin sentimen lagi, cuma narasikan + cross-check)

In [ ]:
def build_narrative_prompt(author_label, theme, texts, sentiment_label, sentiment_confidence):
    joined = "\n".join(f"- {t}" for t in texts)
    return f"""Berikut kumpulan cuitan dari akun "{author_label}" tentang topik "{theme}":

{joined}

Model sentimen sudah menentukan sentimen KESELURUHAN akun ini: "{sentiment_label}"
(confidence: {sentiment_confidence:.2f}).

Tugas kamu:
1. Ringkas dalam 2-3 kalimat apa pandangan/narasi utama akun ini soal topik tersebut.
2. Berikan alasan singkat (1 kalimat) kenapa sentimen "{sentiment_label}" itu masuk akal berdasarkan isi teks.
3. Kalau menurut kamu label sentimen itu SALAH berdasarkan isi teks, sebutkan di field "disagree_reason",
   kalau setuju kosongkan field itu.

Jawab HANYA dengan JSON, tanpa penjelasan tambahan, tanpa markdown code block:
{{"summary": "...", "reason": "...", "disagree_reason": ""}}"""


def generate_narrative(author_label, theme, texts, sentiment_label, sentiment_confidence, retry=3):
    prompt = build_narrative_prompt(author_label, theme, texts, sentiment_label, sentiment_confidence)
    for attempt in range(retry):
        try:
            return gemini_json(prompt, 1000)
        except Exception as e:
            if is_quota_error(e):
                raise
            wait = 5 * (attempt + 1)
            print(f"[WARN] gagal narrative {author_label} ({attempt+1}/{retry}): {e} -> tunggu {wait}s")
            time.sleep(wait)
    return {"summary": "(gagal diringkas otomatis)", "reason": "error API", "disagree_reason": ""}


results = []
quota_hit_summary = False

for _, row in df_top_authors.iterrows():
    author_id, theme = row["author_id"], row["theme"]
    key = (author_id, theme)
    if key not in author_sentiment_results:
        continue

    data = author_sentiment_results[key]
    print(f"Generate narasi @{author_id} | tema={theme} | sentimen={data['sentiment']} ...")

    try:
        narrative = generate_narrative(author_id, theme, data["posts"], data["sentiment"], data["confidence"])
    except Exception as e:
        if is_quota_error(e):
            quota_hit_summary = True
            break
        raise

    if narrative.get("disagree_reason"):
        print(f"  [FLAG] Gemini tidak setuju sentimen IndoBERT: {narrative['disagree_reason']}")

    results.append({
        "theme": theme,
        "rank": int(row["rank"]),
        "author_id": author_id,
        "score": round(row["score"], 2),
        "sentiment": data["sentiment"],
        "sentiment_confidence": round(data["confidence"], 2),
        "summary": narrative.get("summary", ""),
        "reason": narrative.get("reason", ""),
        "gemini_disagree": narrative.get("disagree_reason", ""),
    })

df_summary = pd.DataFrame(results)

if quota_hit_summary:
    print(f"\n[STOP] Berhenti di tengah karena quota habis. {len(results)} author sudah beres.")
else:
    print(f"\n[DONE] Semua author beres. Total API call sesi ini: {api_call_count}")

print("\n=== Hasil akhir: ringkasan & sentimen per top author ===")
display(df_summary)